# Fusion pipeline demo

Pairs an audio clip with a dialogue, fits the fusion models on train, and compares them on validation
under clean and degraded conditions. It runs end to end on dummy tables when the real score files are
not all there, and a banner says so.

**What each setting does** (all are in the next cell, all default to the safe value):

- `RUN_TEST_SPLIT` loads and evaluates the test split. Off by default: test rows are dropped while the files
  are read, so they never enter memory. If it is on but the test scores are missing, the banner says the test
  pairs cannot be built and why.
- `RUN_EXTERNAL` does nothing yet. **In-the-Wild is skipped even when it is on.** Planned design (not built):
  each clean test pair has its audio clip swapped for an In-the-Wild clip with the same audio label; the dialogue,
  text score and target stay unchanged.
- `SAVE_FIGURES` saves PNGs only into a git-ignored folder. **This notebook never writes into outputs/**; it only
  reads score tables from there. Figures are shown inline and held in memory.
- `SEED` is used for loading, pairing and the bootstrap, and is shown in the banner.

**Reading the flags.** Every figure title and the results banner show `dummy=` and `plumbing_only=` taken from the
results table. `dummy=True` means a dummy or smoke input. `plumbing_only=True` means the train scores could not be
verified out-of-fold (no `fold` column). Neither is a finding.

In [ ]:
RUN_TEST_SPLIT = False   # True also loads and evaluates the test split
RUN_EXTERNAL = False     # True changes nothing yet: In-the-Wild pairing is not built
SAVE_FIGURES = False     # True saves PNGs into a git-ignored folder only
SEED = 0                 # loading, pairing and the bootstrap
TEXT_MODEL = "minilm"    # "minilm" or "baseline"
AUDIO_RUN = "smoke"      # audio run folder name
AUDIO_MODEL = "wav2vec2" # "wav2vec2" or "mfcc"
LOWER_IS_WORSE = False   # True: SNR-style noise severities run clean to lowest SNR
N_BOOT = 1000            # bootstrap resamples per interval
DATA_ROOT = None         # None means the project folder; otherwise a folder holding the score-table folders

In [ ]:
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, Image, Markdown, display

project_dir = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "datasets.py").exists())
sys.path.insert(0, str(project_dir))
data_root = project_dir if DATA_ROOT is None else Path(DATA_ROOT)

from src.fusion.checks import check_pairs
from src.fusion.dummy import degrade_scores
from src.fusion.evaluate import evaluate, weight_ratio_sweep
from src.fusion.models import check_inputs_match, fit_fusion, load_model, save_model
from src.fusion.pairing import build_pairs, source_counts
from src.fusion.plots import (
    assert_flags_match,
    banner_html,
    banner_lines,
    condition_label,
    figure_png,
    flag_label,
    plot_difference,
    plot_metric_vs_severity,
    plot_weight_sweep,
    save_figure,
)
from src.fusion.sources import load_sources

## 1. Load the score tables

The loader reads the audio and text score files under the project folder and falls back to dummy tables if
either is missing. Read the banner before anything else.

In [ ]:
bundle = load_sources(data_root, audio_run=AUDIO_RUN, audio_model=AUDIO_MODEL, text_model=TEXT_MODEL,
                      include_test=RUN_TEST_SPLIT, seed=SEED)
display(HTML(banner_html(banner_lines(bundle, run_test_split=RUN_TEST_SPLIT, run_external=RUN_EXTERNAL,
                                      require_oof=bundle.has_folds, seed=SEED))))
display(pd.DataFrame({"rows": {"audio": len(bundle.audio), "text": len(bundle.text)}}))

## 2. Build pairs and check for leakage

Pairs are built inside each split, so no clip or dialogue crosses splits. `check_pairs` re-checks the manifest
against the score tables and raises with every violated rule.

In [ ]:
pairs = build_pairs(bundle.audio, bundle.text, dummy=bundle.dummy, seed=SEED)
check_pairs(pairs, bundle.audio, bundle.text)
display(Markdown("**Source counts per split** (unique clips and dialogues, largest reuse of each)"))
display(source_counts(pairs))
if RUN_TEST_SPLIT and not (pairs["split"] == "test").any():
    reason = bundle.test_unavailable_reason or "no test pairs came out of pairing; see the warnings above"
    display(Markdown(f"**Test split requested but no test pairs were built:** {reason}. No test results are produced."))

## 3. Fit on train, choose C and eps on validation, freeze

The learned rule is a logistic regression on log-scores. C and eps are chosen on validation only; the product rule
is fixed (equal weights, default eps). The model is saved to a temporary folder and reloaded to show that the frozen
copy scores identically. Nothing is written into the project.

In [ ]:
model = fit_fusion(pairs, bundle.audio, bundle.text, require_oof=bundle.has_folds)
with tempfile.TemporaryDirectory() as folder:
    frozen_path = Path(folder) / "model.json"
    save_model(model, frozen_path)
    frozen = load_model(frozen_path)
check_inputs_match(frozen, pairs, bundle.audio, bundle.text)
assert frozen.to_dict() == model.to_dict()

summary = {
    "w_audio (learned)": frozen.weights["audio"],
    "w_text (learned)": frozen.weights["text"],
    "weight ratio w_audio / w_text": frozen.weight_ratio,
    "intercept": frozen.intercept,
    "tuned eps (learned rule)": frozen.eps,
    "default eps (product rule)": frozen.product["eps"],
    "C": frozen.C,
    "fit split / tune split": f"{frozen.fit_split} / {frozen.tune_split}",
    "pairs used to fit / to tune": f"{frozen.n_fit_pairs} / {frozen.n_tune_pairs}",
    "threshold FPR (validation)": frozen.threshold_fpr,
    "oof_verified": frozen.oof_verified,
    "plumbing_only": frozen.plumbing_only,
    "dummy": frozen.dummy,
}
display(pd.DataFrame({"value": summary}))
display(pd.DataFrame({"threshold": frozen.thresholds}))
display(pd.DataFrame({"sha256 (first 16)": {name: digest[:16] for name, digest in frozen.hashes.items()}}))
display(pd.DataFrame(frozen.tuning).round(4))

## 4. Evaluate across conditions

Conditions available: `clean`, plus degraded ones. With dummy tables the degraded conditions are made by
`degrade_scores`, which is test scaffolding and not a real degradation. With real text, the text-prefix conditions
come from the validation prefix scores. Real noise and In-the-Wild audio scores do not exist yet, so those
conditions are simply absent.

In [ ]:
splits = ["validation"] + (["test"] if RUN_TEST_SPLIT and (pairs["split"] == "test").any() else [])


def condition_list(split):
    # (condition, severity, audio table, text table) for every condition available on this split
    conditions = [("clean", None, bundle.audio, bundle.text)]
    if bundle.audio_origin == "dummy":
        for k, (severity, shrink) in enumerate([(20, 0.8), (10, 0.5), (5, 0.3)]):
            weaker = degrade_scores(bundle.audio, shrink, 0.5, seed=SEED + 1 + k)
            conditions.append(("noise", severity, weaker, bundle.text))
        for k, fraction in enumerate([0.75, 0.5, 0.25]):
            shorter = degrade_scores(bundle.text, 0.4 + 0.6 * fraction, 0.5, seed=SEED + 10 + k)
            conditions.append(("text_prefix", fraction, bundle.audio, shorter))
    elif split == "validation":
        for fraction, table in sorted(bundle.prefix_tables.items(), reverse=True):
            conditions.append(("text_prefix", fraction, bundle.audio, table))
    return conditions


frames = []
for split in splits:
    for condition, severity, audio_table, text_table in condition_list(split):
        frames.append(evaluate(frozen, pairs, audio_table, text_table, condition=condition, severity=severity,
                               split=split, allow_test=RUN_TEST_SPLIT, seed=SEED, n_boot=N_BOOT))
results = pd.concat(frames, ignore_index=True)
results["condition_label"] = [condition_label(c, s) for c, s in zip(results["condition"], results["severity"])]

flag = flag_label(results)
assert_flags_match(bundle.dummy, not bundle.has_folds, results)
n_conditions = results[["split", "condition_label"]].drop_duplicates().shape[0]
display(HTML(banner_html([f"Results: {flag}",
                          f"{len(results)} result rows over {n_conditions} split and condition combinations "
                          f"(splits: {', '.join(splits)}); seed {SEED}; {N_BOOT} bootstrap resamples."])))

### Results table

The same numbers the figures show. Intervals are 95% percentile intervals from a bootstrap that resamples whole
dialogues (a dialogue appears in more than one pair).

In [ ]:
shown = results[results["metric"].isin(["roc_auc", "pr_auc"])]
display(shown.pivot_table(index=["split", "condition_label"], columns=["metric", "system"], values="estimate",
                          sort=False).round(3))
rates = results[results["metric"].isin(["tpr_at_threshold", "fpr_at_threshold"])]
display(Markdown("**TPR and FPR at each system's frozen validation threshold (5% FPR on validation)**"))
display(rates.pivot_table(index=["split", "condition_label"], columns=["metric", "system"], values="estimate",
                          sort=False).round(3))

## 5. Figures

Colours are the Okabe-Ito colour-blind-safe set; each system also has its own line style and marker. Shaded bands
and bars are 95% intervals. Every title carries the dummy and plumbing flags from the results.

In [ ]:
def show(fig, name):
    display(Image(figure_png(fig)))
    if SAVE_FIGURES:
        save_figure(fig, name, project_dir / "data" / "fusion_figures", repo_root=project_dir)


for split in splits:
    part = results[results["split"] == split]
    if (part["condition"] == "clean").all():
        display(Markdown(f"**{split}: no degraded conditions are available, so there is nothing to plot "
                         "against severity.**"))
        continue
    show(plot_metric_vs_severity(part, "roc_auc", flag=flag, lower_is_worse=LOWER_IS_WORSE), f"roc_auc_{split}")
    show(plot_metric_vs_severity(part, "pr_auc", flag=flag, lower_is_worse=LOWER_IS_WORSE), f"pr_auc_{split}")
    show(plot_difference(part, flag=flag, lower_is_worse=LOWER_IS_WORSE), f"learned_minus_product_{split}")

## 6. Weight-ratio sweep

AUC of `w_audio * log(audio) + w_text * log(text)` as the ratio `w_audio / w_text` varies, per condition. It uses the
default eps (the same as the product rule), not the tuned eps. Nothing is tuned or selected from it.

In [ ]:
display(Markdown(f"Sweep eps: default eps = {frozen.product['eps']:g} (the product rule's); the learned rule "
                 f"used its tuned eps = {frozen.eps:g}. Learned ratio w_audio / w_text = {frozen.weight_ratio:.3f}."))
sweeps = pd.concat([
    weight_ratio_sweep(pairs, audio_table, text_table, condition=condition, severity=severity, split=split,
                       allow_test=RUN_TEST_SPLIT)
    for split in splits
    for condition, severity, audio_table, text_table in condition_list(split)
], ignore_index=True)
for split in splits:
    show(plot_weight_sweep(sweeps[sweeps["split"] == split], frozen.weight_ratio, flag=flag,
                           lower_is_worse=LOWER_IS_WORSE), f"weight_sweep_{split}")